# Rapid report authoring in a notebook

Design and run the actual decorated report against pandas DataFrames before publishing it. This path needs no PostgreSQL, filesystem store, S3, or MinIO; prototype artifacts remain in memory.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from runbook.sdk import load_profiles, plot_line, prototype_report, report, required_aliases
from runbook.sdk.layout import Link, Report
from runbook.sdk.table_style import (
    action,
    condition,
    format_percent,
    rhs_literal,
    rule,
    table_style,
    target_columns,
)

In [ ]:
profile = load_profiles(Path("data/contract/report_profiles.json"))["volatility_demo"]
profile

In [ ]:
prices = pd.read_csv("data/fixtures/daily_prices.csv")
prices.head()

Define the same aliases, calculations, and page layout that can later move into a report module unchanged. The `ALIASES` and decorated functions can move there unchanged when ready for production.

`prices` is only a Python variable; `Ctx` never discovers variable names. The explicit `frames={"prices": prices}` binding associates that DataFrame with the stable report alias `"prices"`.

Prototype mode freezes the frame into an immutable manifest and stores its alias-to-manifest reference in `Snapshot.datasets["prices"]`. In production, `profile.datasets["prices"]` maps the same report alias to a production dataset ID; pointer/manifest resolution builds `Snapshot.datasets["prices"]`.

In both modes, `ctx.dataset(ALIASES.prices)` resolves `"prices"` through the Snapshot to the immutable DataFrame, so the report functions stay unchanged.

`frames` → `Snapshot.datasets["prices"]` → `ctx.dataset(ALIASES.prices)` → immutable DataFrame

In [ ]:
ALIASES = required_aliases(prices="prices")


@report.calc("returns")
def returns(ctx):
    """Calculate close-to-close returns for the configured price dataset."""
    df = ctx.dataset(ALIASES.prices).copy()
    params = ctx.config.get("params", {})
    price_col = str(params.get("price_col", "price"))
    if price_col not in df.columns:
        raise ValueError(f"price column is missing from dataset: {price_col!r}")
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="raise")
        df = df.sort_values("timestamp", kind="mergesort").set_index("timestamp")
    s = df[price_col].astype(float).pct_change(fill_method=None)
    return s.rename("returns").to_frame()

In [ ]:
@report.calc("vol")
def vol(ctx):
    """Calculate rolling annualized volatility using report parameters."""
    rets = ctx.calc("returns")["returns"]
    params = ctx.config.get("params", {})
    window = int(params.get("vol_window", 20))
    v = rets.rolling(window=window).std(ddof=0).mul(pd.Series(260.0**0.5, index=rets.index))
    return v.rename("vol").to_frame()

In [ ]:
@report.page
def page(ctx):
    """Build the range-volatility demonstration page."""
    returns = ctx.calc("returns")
    vol = ctx.calc("vol")
    layout = ctx.config.get("layout", {})
    width = int(layout.get("plot_width", 1000))
    height = int(layout.get("plot_height", 450))
    fig_returns = plot_line(
        data=returns,
        title="Returns",
        width=width,
        height=height,
        show_legend=False,
        use_rangebreaks=True,
    )
    fig_vol = plot_line(
        data=vol,
        title="Volatility",
        width=width,
        height=height,
        show_legend=False,
        use_rangebreaks=True,
        series_styles={"vol": {"line": {"color": "#2A6F9E"}}},
    )

    # Keep timestamp indexes for deterministic calculations and plots, while
    # materializing JSON-safe timestamp values for the table identity/hash.
    def table_frame(frame):
        """Return a display copy with timestamp indexes rendered as strings."""
        if not isinstance(frame.index, pd.DatetimeIndex):
            return frame
        display = frame.reset_index()
        display["timestamp"] = display["timestamp"].astype(str)
        return display

    # Example deterministic table styler artifacts (`table-style/0.1`) via SDK builders.
    returns_style = table_style(
        key="returns_style_v1",
        formats=[format_percent("returns", digits=2)],
        rules=[
            rule(
                "neg_returns_red",
                target_columns(["returns"]),
                condition("lt", rhs=rhs_literal(0)),
                action(text_color="#B00020", font_weight="600"),
            )
        ],
        max_rows=100,
        na_rep="-",
    )
    vol_style = table_style(
        key="vol_style_v1",
        formats=[format_percent("vol", digits=2)],
        rules=[
            rule(
                "high_vol_yellow",
                target_columns(["vol"]),
                condition("gt", rhs=rhs_literal(0.3)),
                action(background_color="#FFF3CD"),
            )
        ],
        max_rows=100,
        na_rep="-",
    )

    returns_ref = ctx.artifact.table(table_frame(returns), name="returns", style=returns_style)
    vol_ref = ctx.artifact.table(table_frame(vol), name="vol", style=vol_style)
    returns_plot_ref = ctx.artifact.plot(fig_returns, name="returns")
    vol_plot_ref = ctx.artifact.plot(fig_vol, name="vol")

    layout = Report(ctx.config.get("title", "Range Vol (PoC)"))
    with layout.row(columns=2) as returns_row:
        with returns_row.stack() as returns_stack:
            returns_stack.table(returns_ref, name="returns_table", title="Returns")
            returns_stack.add(Link("Visit example.com →", url="https://example.com", name="example-link"))
        returns_row.plot(returns_plot_ref, name="returns_plot", title="Returns Plot")
    with layout.row(columns=2) as vol_row:
        vol_row.table(vol_ref, name="vol_table", title="Volatility")
        vol_row.plot(vol_plot_ref, name="vol_plot", title="Volatility Plot")
    return layout

In [ ]:
observed_at = datetime(2026, 1, 31, 12, tzinfo=timezone.utc)
result = prototype_report(
    profile=profile,
    frames={"prices": prices},
    calculations={"returns": returns, "vol": vol},
    page=page,
    observed_at=observed_at,
)
result

The prototype builds and publishes the Stage 3, Stage 4, and HTML artifacts in memory. When ready for production publication, move `ALIASES` and these decorated functions into a report module unchanged.

In [ ]:
assert "returns" in result.cache_hits
assert "vol" in result.cache_hits
assert result.stage3_ref.endswith("manifest.stage3.json")
assert result.stage4_ref.endswith("manifest.stage4.json")
assert result.html_ref.endswith("report.html")
{
    "cache_hits": result.cache_hits,
    "stage3_ref": result.stage3_ref,
    "stage4_ref": result.stage4_ref,
    "html_ref": result.html_ref,
}